In [2]:
# 配置环境变量
from dotenv import load_dotenv
from openai import conversations

load_dotenv()

import os
import sys
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")
if not api_key or api_key == 'your_dashscope_api_key_here':
    raise ValueError("请将DashScope API Key填写到.env文件中")
model = init_chat_model(
    model="qwen-max",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

# 1.定义简单的工具

In [3]:
@tool
def get_user_info(user_id: str) -> str:
    """
    获取用户信息

    参数
        user_id：用户id

    返回值
        字符串
    """
    user = {
        "123": "张三，25岁，程序员",
        "456": "李四，40岁，设计师",
    }
    return user.get(user_id, "用户不存在")

# 2.客服机器人

In [12]:
def example():
    agent = create_agent(
        model=model,
        tools=[get_user_info],
        system_prompt="""
        你是一个客服助手。
        特点：
        - 记住用户说过的话
        - 友好、有耐心
        - 使用 get_user_info 工具查询用户信息时需要用户 ID
        """,
        checkpointer=InMemorySaver()
    )

    user_id="user_12345"
    config = {"configurable": {"thread_id": user_id}}

    conversations = [
        "你好，我想咨询一下",
        "用户id为123的名字是什么？",
        "他年龄多大来着？",
        "设计师身份的是哪个用户？"
    ]

    for i,msg in enumerate(conversations, 1):
        print(f"{i}. {msg}")
        response = agent.invoke({
                "messages": [{"role": "user", "content": msg}]
            },config=config)
        print(f"客服： {response["messages"][-1].content}")

In [13]:
def main():
    print("\n" + "="*70)
    print("Memory_basic")
    print("="*70)

    try:
        example()
    except KeyboardInterrupt:
        print("\n程序中断")
    except Exception as e:
        print(f"错误： {e}")
        import traceback
        traceback.print_exc()


In [14]:
if __name__ == "__main__":
    main()


Memory_basic
1. 你好，我想咨询一下
客服： 你好！很高兴为您服务。请问您想咨询什么问题呢？
2. 用户id为123的名字是什么？
客服： 用户ID为123的用户名字是张三。
3. 他年龄多大来着？
客服： 用户ID为123的用户张三，他的年龄是25岁。
4. 设计师身份的是哪个用户？
客服： 用户ID为456的用户李四是设计师身份。如果还有其他用户需要查询，请告诉我他们的用户ID。
